첫번째 실습에서 우리는 풀 파인튜닝(Full Fine-tuning)이 뛰어난 정확도를 내지만, 모델의 모든 매개변수(100%)를 학습해야 하므로 컴퓨팅 자원과 저장공간 소모가 극심함을 배웠습니다.

이번 2번재 실습에서는 이러한 문제를 극복하기 위한 대표적인 **PEFT (Parameter-Efficient Fine-Tuning)** 기법인 **LoRA (Low-Rank Adaptation)** 의 개념을 배우고, 직접 모델에 적용하여 학습을 수행해 봅니다.

## 1. PEFT와 LoRA의 이해

### 1) PEFT (Parameter-Efficient Fine-Tuning) 란?
- 사전 훈련된 거대 모델(Base Model)의 대부분 가중치는 **동결(Freeze)** 하고, 매우 일부분의 매개변수만 추가하거나 튜닝하여 새로운 작업에 적응시키는 기법입니다.
- **장점**:
  - **VRAM 소모 급감**: 최적화 기구(Adam 등)가 관리해야 할 파라미터가 크게 줄어 GPU 메모리를 훨씬 적게 씁니다.
  - **저장용량 절약**: 전체 모델 파일(예: 300MB~수십GB)을 태스크마다 따로 저장할 필요 없이, 몇십 KB ~ 몇 MB 크기의 가중치(어댑터) 파일만 관리하면 됩니다.
  - **치명적 망각 방지**: 기존 파라미터를 그대로 보존하므로 일반 성능이 유지됩니다.

### 2) LoRA (Low-Rank Adaptation) 의 작동 원리
- 모델이 학습할 가중치 업데이트 행렬 $\Delta W$를 직접 훈련하는 대신, 저차원의 두 개의 행렬 $A$와 $B$의 곱으로 **분해(Decomposition)** 하여 학습시키는 기법입니다.
  $$\Delta W \approx B \cdot A$$
  - 여기서 원래 가중치가 $d \times k$ 차원이고 저차원 랭크가 $r$ ($r \ll d, k$) 일 때:
    - $B \in \mathbb{R}^{d \times r}$ (0으로 초기화)
    - $A \in \mathbb{R}^{r \times k}$ (가우시안 분포로 초기화)
- 결과적으로 가중치 매개변수의 수는 $(d \times k)$ 에서 $r \times (d + k)$ 로 압도적으로 줄어듭니다.